In [1]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from app.services.anpr_service import recognize_plate_ocr_only
from app.utils.evaluate import edit_distance, normalize_plate, character_accuracy

Using CPU. Note: This module is much faster with a GPU.


In [2]:
IMAGE_DIR = PROJECT_ROOT / "data/samples/cropped"
LABELS_PATH = IMAGE_DIR / "labels.csv"

labels_df = pd.read_csv(LABELS_PATH)
labels_df.drop(columns=["Country"], inplace=True)
labels_df["true_plate"] = labels_df["Plate text"].apply(normalize_plate)

labels_df

,File,Plate text,true_plate
0,plate_03.png,OS BY 153,OSBY153
1,plate_01.png,SHA OZ 268,SHAOZ268
2,plate_04.png,HH NB 1402,HHNB1402
3,plate_05.png,OF PM 550,OFPM550
4,plate_06.png,M EA 4903,MEA4903
5,plate_02.png,DU BE 102 E,DUBE102E
6,plate_07.png,NOH MV 55,NOHMV55
7,plate_08.png,HOM SZ 227,HOMSZ227
8,plate_09.png,M KR 2125,MKR2125
9,plate_10.png,HD SK 630,HDSK630


In [3]:
results = []

for _, row in labels_df.iterrows():
    filename = row["File"]
    true_plate = row["true_plate"]

    image_path = IMAGE_DIR / filename

    with open(image_path, "rb") as f:
        image_bytes = f.read()

    try:
        result = recognize_plate_ocr_only(
            image_bytes=image_bytes,
            filename=filename,
            content_type="image/png",
        )

        predicted_plate = normalize_plate(result["plate_text"])
        raw_text = result.get("raw_text", "")

        exact_match = predicted_plate == true_plate
        char_acc = character_accuracy(predicted_plate, true_plate)

        results.append({
            "filename": filename,
            "true_plate": true_plate,
            "predicted_plate": predicted_plate,
            "raw_text": raw_text,
            "exact_match": exact_match,
            "character_accuracy": char_acc,
            "confidence": result["confidence"],
            "valid_format": result["valid_format"],
            "latency_ms": result["latency_ms"],
            "engine": result["engine"],
            "error": None,
        })

    except Exception as e:
        results.append({
            "filename": filename,
            "true_plate": true_plate,
            "predicted_plate": "",
            "raw_text": "",
            "exact_match": False,
            "character_accuracy": 0.0,
            "confidence": 0.0,
            "valid_format": False,
            "latency_ms": None,
            "engine": "easyocr_ocr_only_v1",
            "error": str(e),
        })

results_df = pd.DataFrame(results)

/home/trieu/projects_linux/ANPR-evalsys/.venv/lib/python3.10/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12050). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
/home/trieu/projects_linux/ANPR-evalsys/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TEXT: OSSBY
CONF: 0.7680088666515269
BBOX: [[np.int32(9), np.int32(21)], [np.int32(251), np.int32(21)], [np.int32(251), np.int32(90)], [np.int32(9), np.int32(90)]]

TEXT: 153
CONF: 0.9999938745316418
BBOX: [[np.float64(260.0), np.float64(21.0)], [np.float64(388.0), np.float64(6.0)], [np.float64(394.0), np.float64(56.0)], [np.float64(265.0), np.float64(70.0)]]

TEXT: 02
CONF: 0.9801697471662146
BBOX: [[np.int32(160), np.int32(23)], [np.int32(245), np.int32(23)], [np.int32(245), np.int32(71)], [np.int32(160), np.int32(71)]]

TEXT: SHA
CONF: 0.9996648960887257
BBOX: [[np.float64(10.0), np.float64(4.0)], [np.float64(124.0), np.float64(17.0)], [np.float64(119.0), np.float64(61.0)], [np.float64(5.0), np.float64(48.0)]]

TEXT: 268
CONF: 0.9999980728858172
BBOX: [[np.float64(250.0), np.float64(28.0)], [np.float64(373.0), np.float64(41.0)], [np.float64(368.0), np.float64(87.0)], [np.float64(245.0), np.float64(75.0)]]

TEXT: HARB
CONF: 0.47582578659057617
BBOX: [[0, np.int32(3)], [np.int32(208),

In [4]:
results_df

,filename,true_plate,predicted_plate,raw_text,exact_match,character_accuracy,confidence,valid_format,latency_ms,engine,error
0,plate_03.png,OSBY153,OSSBY153,OSSBY 153,False,0.875000,0.8840,True,371,easyocr_ocr_only_v1,None
1,plate_01.png,SHAOZ268,SHAOZ268,SHA 02 268,True,1.000000,0.9933,True,273,easyocr_ocr_only_v1,None
2,plate_04.png,HHNB1402,HARB112,HARB 112,False,0.500000,0.4888,True,328,easyocr_ocr_only_v1,None
3,plate_05.png,OFPM550,OFPM55O,OF PM55O,False,0.857143,0.7606,False,305,easyocr_ocr_only_v1,None
4,plate_06.png,MEA4903,HREA5905,HREA 5905,False,0.500000,0.1498,True,290,easyocr_ocr_only_v1,None
5,plate_02.png,DUBE102E,DUBE02,DU BE J02E,False,0.750000,0.8093,True,255,easyocr_ocr_only_v1,None
6,plate_07.png,NOHMV55,NOHMV55,NOH-MV 55,True,1.000000,0.5097,True,232,easyocr_ocr_only_v1,None
7,plate_08.png,HOMSZ227,HOMSZ227,HOM SZ 227,True,1.000000,0.9928,True,264,easyocr_ocr_only_v1,None
8,plate_09.png,MKR2125,MOKR2125,MOKR 2125,False,0.875000,0.7969,True,258,easyocr_ocr_only_v1,None
9,plate_10.png,HDSK630,HDSK630,HD SK 630,True,1.000000,0.9991,True,322,easyocr_ocr_only_v1,None


In [5]:
num_samples = len(results_df)

exact_accuracy = results_df["exact_match"].mean()
avg_character_accuracy = results_df["character_accuracy"].mean()
avg_confidence = results_df["confidence"].mean()
avg_latency = results_df["latency_ms"].dropna().mean()
valid_format_rate = results_df["valid_format"].mean()

summary = {
    "num_samples": num_samples,
    "exact_plate_accuracy": exact_accuracy,
    "avg_character_accuracy": avg_character_accuracy,
    "avg_confidence": avg_confidence,
    "avg_latency_ms": avg_latency,
    "valid_format_rate": valid_format_rate,
}

summary

{'num_samples': 10,
 'exact_plate_accuracy': np.float64(0.4),
 'avg_character_accuracy': np.float64(0.8357142857142857),
 'avg_confidence': np.float64(0.73843),
 'avg_latency_ms': np.float64(289.8),
 'valid_format_rate': np.float64(0.9)}